### Các bộ phân được tách
head 0

shoulder 11, 12

elbow 13, 14

wrist 15, 16

hip 23, 24

knee 25, 26

ankle 27, 28

In [ ]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

video_path = "Test2.mp4"

base_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1
    )

group = {
    0: 'wrist',
    1: 'thumb_cmc',
    2: 'thumb_mcp',
    3: 'thumb_ip',
    4: 'thumb_tip',
    5: 'index_finger_mcp',
    6: 'index_finger_pip',
    7: 'index_finger_dip',
    8: 'index_finger_tip',
    9: 'middle_finger_mcp',
    10: 'middle_finger_pip',
    11: 'middle_finger_dip',
    12: 'middle_finger_tip',
    13: 'ring_finger_mcp',
    14: 'ring_finger_pip',
    15: 'ring_finger_dip',
    16: 'ring_finger_tip',
    17: 'pinky_mcp',
    18: 'pinky_pip',
    19: 'pinky_dip',
    20: 'pinky_tip',
}

keypoints = [0, 4, 8, 12, 16, 20]

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

def draw_landmarks(frame, result):
    if not result.hand_landmarks:
        return frame
    
    frame_copy = frame.copy()
    (h, w) = frame_copy.shape[:2]

    point_color = (0, 0, 255)

    for idx in keypoints:
        
        landmark = result.hand_landmarks[0][idx]
        
        px = int(landmark.x * w)
        py = int(landmark.y * h)

        cv2.circle(frame_copy, (px, py), 6, point_color, -1)

    return frame_copy

extracted_data = []
def extract_data(result, frame_idx):
    if not result.hand_world_landmarks:
        return
    
    data = result.hand_world_landmarks[0]

    for landmark_id, landmark in enumerate(data):
        if landmark_id not in keypoints:
            continue

        extracted_data.append({
            'frame': frame_idx,
            'part': group[landmark_id],
            'part_idx': landmark_id,
            'x': landmark.x,
            'y': landmark.y,
            'z': landmark.z
        })

with HandLandmarker.create_from_options(base_options) as landmarker:
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        timestamp_ms = int((frame_idx / fps) * 1000)
        image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        result = landmarker.detect_for_video(image, timestamp_ms)
        
        if result.hand_landmarks:
            for landmark_id, landmark in enumerate(result.hand_landmarks[0]):
                print(f"Frame {frame_idx} | Landmark #{landmark_id}\nx={landmark.x:.3f}\ny={landmark.y:.3f}\nz={landmark.z:.3f}\n")
        
        drawed_frame = draw_landmarks(frame, result)

        cv2.imshow("Camera", drawed_frame)

        extract_data(result, frame_idx)
        
        frame_idx += 1

cap.release()
cv2.destroyAllWindows()

Frame 0 | Landmark #0
x=0.603
y=0.462
z=-0.000

Frame 0 | Landmark #1
x=0.503
y=0.552
z=-0.004

Frame 0 | Landmark #2
x=0.420
y=0.616
z=-0.018

Frame 0 | Landmark #3
x=0.363
y=0.689
z=-0.035

Frame 0 | Landmark #4
x=0.319
y=0.755
z=-0.054

Frame 0 | Landmark #5
x=0.390
y=0.482
z=-0.031

Frame 0 | Landmark #6
x=0.336
y=0.630
z=-0.064

Frame 0 | Landmark #7
x=0.352
y=0.723
z=-0.082

Frame 0 | Landmark #8
x=0.371
y=0.756
z=-0.092

Frame 0 | Landmark #9
x=0.436
y=0.482
z=-0.046

Frame 0 | Landmark #10
x=0.378
y=0.647
z=-0.075

Frame 0 | Landmark #11
x=0.392
y=0.746
z=-0.081

Frame 0 | Landmark #12
x=0.409
y=0.767
z=-0.087

Frame 0 | Landmark #13
x=0.486
y=0.502
z=-0.062

Frame 0 | Landmark #14
x=0.431
y=0.655
z=-0.091

Frame 0 | Landmark #15
x=0.438
y=0.748
z=-0.089

Frame 0 | Landmark #16
x=0.452
y=0.770
z=-0.086

Frame 0 | Landmark #17
x=0.535
y=0.532
z=-0.078

Frame 0 | Landmark #18
x=0.490
y=0.662
z=-0.100

Frame 0 | Landmark #19
x=0.479
y=0.727
z=-0.101

Frame 0 | Landmark #20
x=0.480

NameError: name 'GROUP' is not defined

: 

In [3]:
df = pd.DataFrame(extracted_data)
df

""


In [ ]:
df.to_csv("tremor_data.csv")